## PARTIE 2 : Deep Learning - Classification d'images

In [ ]:
import keras
import numpy as np
import pandas as pd
from keras.src.legacy.backend import dropout
from sklearn.preprocessing import StandardScaler

fichier = "WangSignatures.xlsx"

noms_onglets = {
    "JCD": "WangSignaturesJCD",
    "PHOG": "WangSignaturesPHOG",
    "CEDD": "WangSignaturesCEDD",
    "FCTH": "WangSignaturesFCTH",
    "FuzzyColor": "WangSignaturesFuzzyColorHistogr"
}

# Chargement des cinq descripteurs
descripteurs = {}

for nom, onglet in noms_onglets.items():
    df = pd.read_excel(
        fichier,
        sheet_name=onglet,
        header=None,
        index_col=0
    )

    descripteurs[nom] = df

    print(f"{nom} : {df.shape}")

# Vérification des noms d'images
ordre_images = descripteurs["JCD"].index

assert len(ordre_images) == 1000
assert ordre_images.is_unique

for nom, df in descripteurs.items():
    assert df.index.is_unique
    assert len(df) == 1000
    assert set(df.index) == set(ordre_images)

    descripteurs[nom] = df.loc[ordre_images]

# Construction des labels
labels = np.array([
    int(nom[:-4]) // 100
    for nom in ordre_images
])

print("\nRépartition des classes :")
classes, effectifs = np.unique(labels, return_counts=True)

for classe, effectif in zip(classes, effectifs):
    print(f"Classe {classe} : {effectif} images")

# Conversion en matrices NumPy
X_descripteurs = {}

for nom, df in descripteurs.items():
    X_descripteurs[nom] = df.to_numpy(dtype=float)

# Concaténation des cinq descripteurs
X_complet = np.concatenate(
    list(X_descripteurs.values()),
    axis=1
)

# Vérifications finales
assert X_complet.shape[0] == len(labels)
assert np.isfinite(X_complet).all()

print("\nDimensions des matrices :")

for nom, X in X_descripteurs.items():
    print(f"{nom} : {X.shape}")

print(f"\nTous les descripteurs : {X_complet.shape}")
print(f"Labels : {labels.shape}")

## Partie 2.1.2

In [ ]:
from sklearn.model_selection import train_test_split

# ÉTAPE 2 : Séparation stratifiée en train/test (80% / 20%)
indices = np.arange(len(labels))

idx_train_val, idx_test = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels, # Conserve la proportion de chaque classe
    random_state=42
)

# Séparation supplémentaire pour avoir un set de validation (utilisé pour l'EarlyStopping)
idx_train, idx_val = train_test_split(
    idx_train_val,
    test_size=0.2,
    stratify=labels[idx_train_val],
    random_state=42
)

y_train = labels[idx_train]
y_val = labels[idx_val]
y_test = labels[idx_test]

print(f"Images d'entraînement : {len(idx_train)}")
print(f"Images de validation : {len(idx_val)}")
print(f"Images de test : {len(idx_test)}")

In [ ]:
def creer_modele(nb_entrees):
    model = keras.Sequential()
    model.add(keras.Input(shape=(nb_entrees,)))
    model.add(keras.layers.Dense(64, activation="relu"))
    model.add(keras.layers.Dense(32, activation="relu"))
    model.add(keras.layers.Dense(10, activation="softmax"))

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. Création d'un dictionnaire unique regroupant les 6 configurations
ensembles = {**X_descripteurs, "Complet": X_complet}

# Dictionnaire pour stocker les métriques et historiques de chaque modèle
resultats = {}

# 2. Boucle principale d'évaluation
for nom, X in ensembles.items():
    print(f"\n=== Entraînement sur le descripteur : {nom} ===")

    # 3. Extraction des sous-ensembles via les indices stratifiés (générés à l'Étape 2)
    X_train = X[idx_train]
    X_val = X[idx_val]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    model = creer_modele(X_train.shape[1])
    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )
    history = model.fit(
        X_train,
        y_train,
        batch_size=32,
        epochs=100,
        callbacks=[early_stopping],
        validation_data=(X_val, y_val),
        verbose=0
    )

    probas = model.predict(X_val)
    predictions = np.argmax(probas, axis=1)

    acc = accuracy_score(y_val, predictions)
    resultats[nom] = {"acc": acc, "history": history, "model": model, "scaler": scaler}

    print(f"Accuracy validation : {acc:.3f} ({len(history.history['loss'])} époques)")

In [ ]:
import keras
from keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

keras.utils.set_random_seed(42)   # résultats reproductibles

noms_classes = ["Afrique", "Plages", "Monuments", "Bus", "Dinosaures",
                "Éléphants", "Fleurs", "Chevaux", "Montagnes", "Plats"]

In [ ]:
def creer_modele(nb_entrees, couches=(64, 32), dropout=0.0, batch_norm=False, lr=1e-3):
    model = keras.Sequential()
    model.add(keras.Input(shape=(nb_entrees,)))

    for nb_neurones in couches:
        model.add(layers.Dense(nb_neurones, activation="relu"))      # 🔎 nb de neurones de cette couche
        if batch_norm:
            model.add(layers.BatchNormalization())                                   # 🔎 couche de normalisation par lots
        if dropout > 0:
            model.add(layers.Dropout(dropout))                                   # 🔎 couche Dropout avec le taux `dropout`

    model.add(layers.Dense(len(noms_classes), activation="softmax"))           # 🔎 combien de classes ? quelle activation pour des probabilités ?

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",                                          # 🔎 labels ENTIERS 0..9 → quelle crossentropy ?
        metrics=["accuracy"],
    )
    return model

In [ ]:
def entrainer(X, config, verbose=0):
    X_train = X[idx_train]
    X_val = X[idx_val]

    # Normalisation : on apprend la moyenne/écart-type sur le TRAIN uniquement
    scaler = StandardScaler()                                             # 🔎 créer le scaler
    X_train = scaler.fit_transform(X_train)                                            # 🔎 fit + transform
    X_val = scaler.transform(X_val)                                              # 🔎 transform seulement (pourquoi ?)

    model = creer_modele(X_train.shape[1], **config)

    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",                                       # 🔎 quelle grandeur surveiller ?
        patience=15,
        restore_best_weights=True,
    )

    history = model.fit(                                     # 🔎 entraîner
        X_train, y_train,
        batch_size=32,
        epochs=300,
        validation_data=(X_val, y_val),
        callbacks=[early_stopping],
        verbose=verbose,
    )

    probas = model.predict(X_val, verbose=0)                     # 🔎 probabilités (N x 10)
    predictions = np.argmax(probas, axis=1)                                       # 🔎 np.argmax sur quel axe ?
    acc = accuracy_score(y_val, predictions)                                      # 🔎 accuracy sur la validation

    return {"acc": acc, "history": history, "model": model,
            "scaler": scaler, "pred_val": predictions}

In [ ]:
ensembles = {**X_descripteurs, "Complet": X_complet}
config_base = {"couches": (64, 32)}

resultats = {}
for nom, X in ensembles.items():
    resultats[nom] = entrainer(X, config_base)
    r = resultats[nom]
    print(f"{nom:<12} dim={X.shape[1]:<4} acc val = {r['acc']:.3f}  "
          f"({len(r['history'].history['loss'])} époques)")

In [ ]:
tableau = pd.DataFrame({
    "Descripteur": list(resultats.keys()),
    "Dimension": [ensembles[n].shape[1] for n in resultats],
    "Accuracy val": [r["acc"] for r in resultats.values()],
    "Taux d'erreur val": [(1 - r["acc"]) for r in resultats.values()],   # 🔎 à partir de l'accuracy
    "Époques": [len(r["history"].history["loss"]) for r in resultats.values()],
}).sort_values("Accuracy val", ascending=False)                         # 🔎 trier par quelle colonne ?

tableau

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, (nom, r) in zip(axes.flat, resultats.items()):
    h = r["history"].history
    ax.plot(h["loss"], label="apprentissage")                  # 🔎 clé de la loss d'entraînement
    ax.plot(h["val_loss"], label="validation")                     # 🔎 clé de la loss de validation
    ax.set_title(f"{nom} (acc val = {r['acc']:.2f})")
    ax.set_xlabel("Époque")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

Pour tous les descripteurs, la loss d’apprentissage tend vers 0 alors que la loss de validation atteint un plateau, ce qui traduit un surapprentissage, attendu vu le faible nombre d’exemples

In [ ]:
meilleur = max(resultats, key=lambda nom: resultats[nom]["acc"])     # 🔎 nom du descripteur avec la meilleure acc (max sur resultats, key=...)

cm = confusion_matrix(y_val, resultats[meilleur]["pred_val"])                               # 🔎 (vraies classes, prédictions) sur la validation

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm, display_labels=noms_classes).plot(ax=ax, xticks_rotation=45, colorbar=False)
ax.set_title(f"Matrice de confusion (validation) — {meilleur}")
plt.show()

# Taux d'erreur par classe
erreur_par_classe = 1 - np.diag(cm) / cm.sum(axis=1)                             # 🔎 diagonale / somme de chaque LIGNE
for c, e in zip(noms_classes, erreur_par_classe):
    print(f"{c:<10} : {e:.0%} d'erreur")

La concaténation des descripteurs atteint 89 % de bonne classification sur la validation. Les erreurs se concentrent sur les images d'exterieur de nature Plage/montagne , Afrique/elephant

In [ ]:
configs = {
    "petit (32)":            {"couches": (32,)},
    "base (64,32)":          {"couches": (64, 32)},
    "large (256,128)":       {"couches": (256, 128)},
    "large + dropout 0.3":   {"couches": (256, 128), "dropout": 0.3},
    "large + BN":            {"couches": (256, 128), "batch_norm": True},
    "large + BN + dropout":  {"couches": (256, 128), "batch_norm": True, "dropout": 0.3},
    "dropout 0.5":           {"couches": (256, 128), "dropout": 0.5},
    "3 couches":             {"couches": (256, 128, 64), "dropout": 0.3},
    "lr 1e-4":               {"couches": (64, 32), "lr": 1e-4},
    "lr 1e-2":               {"couches": (64, 32), "lr": 1e-2},
}

nom_desc = "Complet"
n_repetitions = 3

lignes = []
for nom_conf, conf in configs.items():
    accs = []
    for graine in range(n_repetitions):
        keras.utils.set_random_seed(graine)
        accs.append(entrainer(ensembles[nom_desc], conf)["acc"])
    lignes.append({"Config": nom_conf,
                   "Acc moyenne": np.mean(accs),                         # 🔎 moyenne de accs
                   "Écart-type": np.std(accs)})                         # 🔎 écart-type de accs
    print(f"{nom_conf:<22} {np.mean(accs):.3f} ± {np.std(accs):.3f}")

pd.DataFrame(lignes).sort_values("Acc moyenne", ascending=False)

In [ ]:
desc_final = "Complet"
config_finale = configs["dropout 0.5"]

keras.utils.set_random_seed(42)
final = entrainer(ensembles[desc_final], config_finale)

X_test = final["scaler"].transform(ensembles[desc_final][idx_test])   # 🔎 même normalisation que le train
pred_test = np.argmax(final["model"].predict(X_test, verbose=0), axis=1)                                                    # 🔎 predict + argmax
acc_test_mlp = accuracy_score(y_test, pred_test)   # 🔎 accuracy sur la validation


print(f"MLP ({desc_final}) — accuracy test : {acc_test_mlp:.3f}")

ConfusionMatrixDisplay(confusion_matrix(y_test, pred_test),
                       display_labels=noms_classes).plot(xticks_rotation=45, colorbar=False)
plt.title("Matrice de confusion (test) — MLP")
plt.show()

In [ ]:
y_tv = labels[idx_train_val]

param_grid = {
    "knn__n_neighbors": [1, 3, 5, 7, 9, 11, 15],
    "knn__metric": ["euclidean", "manhattan"],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

resultats_knn = {}
for nom, X in ensembles.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),                                       # 🔎 normalisation
        ("knn", KNeighborsClassifier()),                                          # 🔎 classifieur KppV
    ])
    grid = GridSearchCV(pipe, param_grid, cv=cv, scoring="accuracy")
    grid.fit(X[idx_train_val], y_tv)                           # 🔎 lancer la recherche

    pred = grid.predict(X[idx_test])                               # 🔎 prédire sur le test
    resultats_knn[nom] = {
        "best_params": grid.best_params_,                               # 🔎 meilleurs paramètres
        "acc_cv": grid.best_score_,                                    # 🔎 meilleur score de validation croisée
        "acc_test": accuracy_score(y_test, pred),
    }
    print(f"{nom:<12} {resultats_knn[nom]['best_params']}  "
          f"cv={resultats_knn[nom]['acc_cv']:.3f}  test={resultats_knn[nom]['acc_test']:.3f}")

In [ ]:
comparaison = []
for nom, X in ensembles.items():
    r = resultats[nom]
    X_test_n = r["scaler"].transform(X[idx_test])
    pred = np.argmax(r["model"].predict(X_test_n, verbose=0), axis=1)
    comparaison.append({
        "Descripteur": nom,
        "MLP test": accuracy_score(y_test, pred),
        "KppV test": resultats_knn[nom]["acc_test"],
        "KppV params": resultats_knn[nom]["best_params"],
    })

pd.DataFrame(comparaison).sort_values("MLP test", ascending=False)